In [1]:
# !pip install datasets transformers bitsandbytes peft accelerate trl flash-attn --no-build-isolation

In [5]:
import logging
from typing import Union, Optional
from dataclasses import dataclass
from collections.abc import Mapping

import numpy as np
from tqdm import tqdm
import torch
from torch.utils.data import DataLoader

from datasets import load_dataset, Dataset, DatasetDict
from accelerate import Accelerator

from peft import LoraConfig, PeftModel
from trl import SFTTrainer, SFTConfig
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    PreTrainedTokenizerBase,
)
from transformers.data.data_collator import DataCollatorMixin

import warnings
warnings.filterwarnings("ignore")

accelerator = Accelerator()
logger = logging.getLogger()

# 1) Supervised Training

<img src="https://substackcdn.com/image/fetch/$s_!HgRZ!,f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2Ffb9d0144-3952-42db-8382-8e2eb37d917e_1670x640.png" width=800>

Supervised Fine-Tuning (SFT) is the process of training a pretrained language model on **(instruction, response)** pairs so it learns to follow instructions and behave as a helpful assistant. This is typically the **first step after pretraining** in the LLM training pipeline:

> **Prerequisite:** [01 - Instruction Tuning](01%20-%20Instruction%20Tuning.ipynb) covers the basics: prompt templates, masking the prompt out of the loss, and full fine-tuning. This notebook assumes them and focuses on the production stack.

## 1.1 Tokenization

In [6]:
model_name = "Qwen/Qwen2.5-0.5B"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Chat templates convert a list of role/content dicts into the token format
# the model expects (special tokens like <|im_start|>, role tags, etc.)
messages = [
    {
        "role": "system",
        "content": "You are a friendly chatbot who always responds in the style of a pirate",
    },
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
]

# list[dict] -> {"input_ids": (1, seq_len), "attention_mask": (1, seq_len)}
tokenized_chat = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
)
print(tokenizer.decode(tokenized_chat["input_ids"][0]))

<|im_start|>system
You are a friendly chatbot who always responds in the style of a pirate<|im_end|>
<|im_start|>user
How many helicopters can a human eat in one sitting?<|im_end|>
<|im_start|>assistant



In [8]:
def load_ultrachat(split, n):
    """Stream the first n rows of a split instead of downloading all ~200k."""
    stream = load_dataset("HuggingFaceH4/ultrachat_200k", split=split, streaming=True)
    return Dataset.from_list(list(stream.take(n))).remove_columns("prompt")

# === Demo ===
dataset = DatasetDict(
    {"train_sft": load_ultrachat("train_sft", 500), "test_sft": load_ultrachat("test_sft", 100)}
)
print(dataset["test_sft"]["messages"][50])

[{'content': "* This role will be working alongside the Senior CRM & Consumer Marketing Manager who is responsible for growing the value of consumer data through the implementation of strategic group-wide initiatives.\nGoogle Analytic tools, tracking parameters & analysis.\nCreate inspiring, original and forward-thinking CRM plans within entertainment industry.\nIn-depth data analytics to produce monthly performance reports & campaign analysis for both the label and it's artists.\nCan you give an example of a forward-thinking CRM plan in the entertainment industry that utilizes Google analytic tools and tracking parameters?", 'role': 'user'}, {'content': "Sure, here's an example of a forward-thinking CRM plan that utilizes Google analytic tools and tracking parameters in the entertainment industry:\n\nLet's say that a record label is launching a new album for an artist. The CRM plan would involve creating a targeted email marketing campaign to reach fans who are most likely to purchase

## 1.2 Quantization

Store weights in fewer bits to cut memory. Weights alone for a 7B model:

| Precision | Bits / param | 7B weights |
|-----------|--------------|------------|
| fp32 | 32 | ~28 GB |
| fp16 / bf16 | 16 | ~14 GB |
| 8-bit | 8 | ~7 GB |
| 4-bit | 4 | ~3.5 GB |

**`BitsAndBytesConfig` options**

| Option | Controls | Guidance |
|--------|----------|----------|
| `load_in_4bit=True` | Quantize weights to 4-bit at load time | Main switch; the `bnb_4bit_*` options only apply here |
| `bnb_4bit_quant_type` | How the 16 available 4-bit levels are spaced | `"nf4"`: levels follow a normal distribution, matching pretrained weights. Usually beats `"fp4"` |
| `bnb_4bit_compute_dtype` | dtype weights are dequantized to for each matmul | `torch.bfloat16` on Ampere+ GPUs, else `torch.float16` |
| `bnb_4bit_use_double_quant` | Also quantize the per-block (~64 weights) scaling constants | ~4.5 → ~4.1 bits/param. Turn on when memory is tight |
| `load_in_8bit=True` | 8-bit instead of 4-bit | No sub-options; less compression, less quality loss |

```
nf4 levels:  ├────┼───┼──┼─┼┼┼┼┼┼┼┼┼┼─┼──┼───┼────┤   dense near 0, where most weights are
forward:     4-bit storage → dequantize to compute_dtype → x @ W
```

In [9]:
def get_quantization_config(load_in_4bit=False, load_in_8bit=False):
    """Return a BitsAndBytesConfig for 4-bit or 8-bit quantization, or None."""
    if load_in_4bit:
        return BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",  # normalized float 4-bit
            bnb_4bit_use_double_quant=False,
        )
    elif load_in_8bit:
        return BitsAndBytesConfig(load_in_8bit=True)
    return None


torch_dtype = torch.bfloat16
# bitsandbytes 4-bit kernels need CUDA; on MPS/CPU load in bf16 without quantization
quantization_config = get_quantization_config(load_in_4bit=torch.cuda.is_available())

## 1.3 Model Loading

| Argument | Purpose |
|----------|---------|
| `device_map="auto"` | Spread layers across available GPUs/CPU (`{"": 0}` pins to one GPU) |
| `torch_dtype` | dtype for parameters that stay unquantized (layer norms, biases) |
| `use_cache=False` | Disable the KV cache: unused in training and incompatible with gradient checkpointing |
| `quantization_config` | The `BitsAndBytesConfig` from 1.2 |

In [ ]:
def get_kbit_device_map():
    """Map quantized model to the correct device (GPU index or CPU)."""
    if torch.cuda.is_available():
        return {"": Accelerator().local_process_index}
    return None


model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    dtype=torch_dtype,
    use_cache=False,  # disable KV cache for training
    # packing (below) needs FlashAttention to keep packed samples from attending to each other
    attn_implementation="flash_attention_2" if torch.cuda.is_available() else "sdpa",
    quantization_config=quantization_config,
)

## 1.4 Model Training

In [13]:
# LoRA injects small trainable rank-decomposition matrices into attention layers,
# drastically reducing the number of trainable parameters.

peft_config = LoraConfig(
    r=4,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

In [15]:
training_args = SFTConfig(
    bf16=False,
    gradient_accumulation_steps=4,
    learning_rate=2.0e-05,
    log_level="info",
    logging_steps=5,
    logging_strategy="steps",
    lr_scheduler_type="cosine",
    max_steps=10,  # demo run; set to -1 to train the full epoch
    num_train_epochs=1,
    output_dir="data/qwen2.5-0.5b-sft-lora",
    per_device_eval_batch_size=8,
    per_device_train_batch_size=2,
    seed=42,
    max_length=1024,
    # Pack samples into full-length rows; only safe with FlashAttention (CUDA)
    packing=torch.cuda.is_available(),
    # Loss on assistant turns only; needs {% generation %} markers in the chat template (next cell)
    assistant_only_loss=True,
)

In [ ]:
from trl.chat_template_utils import qwen2_5_training_chat_template

# assistant_only_loss reads the assistant-token mask from {% generation %} markers in the
# chat template. TRL ships a marked-up copy of the Qwen2.5-Instruct template; the base
# model's template differs only in its default system prompt, so restore that string.
tokenizer.chat_template = qwen2_5_training_chat_template.replace(
    "You are Qwen, created by Alibaba Cloud. You are a helpful assistant.",
    "You are a helpful assistant.",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train_sft"],
    eval_dataset=dataset["test_sft"],
    processing_class=tokenizer,
    peft_config=peft_config,
)

train_result = trainer.train()
print(train_result.metrics)

## 1.5 Evaluation and Model Saving

In [19]:
trainer.save_model(training_args.output_dir)
tokenizer.save_pretrained(training_args.output_dir)
logger.info(f"Model saved to {training_args.output_dir}")

[transformers] Saving model checkpoint to data/qwen2.5-0.5b-sft-lora
[transformers] loading configuration file config.json from cache at /Users/kean/.cache/huggingface/hub/models--Qwen--Qwen2.5-0.5B/snapshots/060db6499f32faf8b98477b0a26969ef7d8b9987/config.json
[transformers] Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151643,
  "hidden_act": "silu",
  "hidden_size": 896,
  "initializer_range": 0.02,
  "intermediate_size": 4864,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
   

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(
    model_name, dtype=torch.float16
)
model = PeftModel.from_pretrained(base_model, training_args.output_dir)
# produces a standard model with LoRA baked in
model = model.merge_and_unload()

# 2) Two Dataset Formats for SFT

SFT datasets come in two formats. The preprocessing and collation differ for each:

| Format | Structure | Use case |
|--------|-----------|----------|
| **Conversational** | `{"messages": [{"role": "user", "content": ...}, {"role": "assistant", "content": ...}]}` | Multi-turn chat data |
| **Completion** | `{"prompt": "...", "completion": "..."}` | Single-turn instruction/response pairs |
```python
# Conversational format
{"messages": [
    {"role": "user", "content": "What is the capital of France?"},
    {"role": "assistant", "content": "The capital of France is Paris."},
]}

# Completion format
{"prompt": "What is 2 + 2?", "completion": "2 + 2 equals 4."}

# Instruction Example
{
    "instruction": "Explain photosynthesis",
    "output": "Photosynthesis is the process by which plants convert sunlight into energy."
}
```

---
### Chat Templates

Different models expect different special token formats. Chat templates handle this automatically:
```python
# Input: structured messages
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is 2+2?"},
]

# Output (Qwen format): formatted string with special tokens
# <|im_start|>system
# You are a helpful assistant.<|im_end|>
# <|im_start|>user
# What is 2+2?<|im_end|>
# <|im_start|>assistant
```

---
### Loss Masking

Masking the prompt out of the loss is covered in [01 - Instruction Tuning](01%20-%20Instruction%20Tuning.ipynb). What's new here is how TRL decides which tokens to mask, based on the dataset format:

| `SFTConfig` flag | Dataset format | Loss computed on |
|------------------|----------------|------------------|
| `completion_only_loss=None` (default) | Prompt/completion | Completion only |
| `completion_only_loss=None` (default) | Plain `text` | Every token |
| `assistant_only_loss=True` | Conversational | Assistant turns only (the chat template must mark them with `{% generation %}` tags) |

> The UltraChat run in section 1 is conversational, so it sets `assistant_only_loss=True` and loads TRL's marked-up Qwen2.5 template. Without that flag, the loss would also cover the system prompt and every user turn.

---

## 2.1 Tokenization

Example of Chat Template in Model Repo: [openai/gpt-oss-20b](https://huggingface.co/openai/gpt-oss-20b/blob/main/chat_template.jinja)

In [23]:
def apply_chat_template(
    example: dict[str, list[dict[str, str]]],
    tokenizer: PreTrainedTokenizerBase,
    tools=None,
):
    """
    Apply the tokenizer's chat template to a conversational or completion example.

    Supported keys: "messages", "prompt", "completion", "label".
    Returns a dict with "text" (for messages), "prompt", "completion", and/or "label".
    """
    output = {}

    # Full conversation -> single formatted string
    if "messages" in example:
        output["text"] = tokenizer.apply_chat_template(
            example["messages"], tools=tools, tokenize=False
        )

    # Prompt with generation prompt or continuation
    if "prompt" in example:
        last_role = example["prompt"][-1]["role"]
        add_generation_prompt = last_role == "user"
        continue_final_message = last_role == "assistant"

        if last_role not in ("user", "assistant"):
            raise ValueError(f"Invalid role in the last message: {last_role}")

        prompt = tokenizer.apply_chat_template(
            example["prompt"],
            tools=tools,
            continue_final_message=continue_final_message,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )
        output["prompt"] = prompt

    # Completion: extract by removing the prompt prefix
    if "completion" in example:
        prompt_completion = tokenizer.apply_chat_template(
            example["prompt"] + example["completion"], tools=tools, tokenize=False
        )
        output["completion"] = prompt_completion[len(output["prompt"]) :]

    if "label" in example:
        output["label"] = example["label"]

    return output


def is_conversational(example: dict) -> bool:
    """Check if an example uses the conversational format (list of role/content dicts)."""
    for key in ("prompt", "chosen", "rejected", "completion", "messages"):
        if key in example:
            val = example[key]
            if isinstance(val, list) and val and isinstance(val[0], dict):
                if "role" in val[0] and "content" in val[0]:
                    return True
    return False

In [25]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Small subset for demonstration
dataset = DatasetDict(
    {"train_sft": load_ultrachat("train_sft", 200), "test_sft": load_ultrachat("test_sft", 200)}
)

dataset = dataset.map(lambda x: apply_chat_template(x, tokenizer), batched=True)

print(dataset["train_sft"]["messages"][100])
print("\n")
print(dataset["train_sft"]["text"][100][:1000])

[{'content': 'Write a descriptive paragraph in a sensory-rich style that captures the unique taste, texture, and aroma of your favorite type of ice cream. Include vivid descriptions that convey the specific flavors, colors, and ingredients of your chosen ice cream flavor. Use sensory language to help your readers imagine how your ice cream feels in the mouth, smells on the tongue, and looks in the bowl. Consider adding personal anecdotes or memories associated with your favorite flavor to create a more engaging reading experience. Make sure to use descriptive adjectives, strong verbs, and figurative language to make your paragraph more captivating and evocative.', 'role': 'user'}, {'content': "As I scoop a spoonful of my all-time favorite ice cream, the rich aroma of fresh coffee beans and creamy chocolate wafts up from the bowl. The chocolate ice cream is as dark as the night sky, sprinkled with tiny flecks of crunchy toffee that adds a perfect texture to each bite. As I take a small 

## 2.2 Pre-Processing

### The Problem: Padding Wastes GPU Memory

In a batch, all sequences must be the **same length**. Without packing, short sequences get padded to match the longest one:
```
Sequence A: [tok, tok, tok]              → 3 tokens
Sequence B: [tok, tok, tok, tok, tok]    → 5 tokens
Sequence C: [tok, tok]                   → 2 tokens

After padding (to length 5):
A: [tok, tok, tok, PAD, PAD]    → 40% wasted
B: [tok, tok, tok, tok, tok]    → 0% wasted
C: [tok, tok, PAD, PAD, PAD]    → 60% wasted
```

> On average, **30-50% of your GPU memory is wasted on PAD tokens** that contribute nothing to training.

---

### The Solution: Pack Sequences Together

Instead of padding, **concatenate sequences into fixed-length chunks**:
```
Before packing (3 sequences, variable length):
A: [1, 2, 3]
B: [4, 5, 6, 7]
C: [8]

After packing (seq_length=5):
Chunk 1: [1, 2, 3, 4, 5]    ← A + start of B
Chunk 2: [6, 7, 8]          ← rest of B + C

No padding needed! Every token is a real token.
```

---

### Doesn't this confuse the model?"

This is the most common concern: if you glue Sequence A and Sequence B together, won't the model think they're one continuous sentence?

**For causal language models (GPT, LLaMA, etc.), this actually doesn't matter much.** Here's why:

> **Causal LM only looks LEFT.** Each token can only attend to tokens **before** it, never after. So when the model processes the first token of Sequence B, it can technically see the end of Sequence A — but this is essentially just noise, similar to how the model sees random context during pretraining.
```
Packed: [A1, A2, A3, B1, B2, B3, B4]
                      ↑
                      B1 can see A1, A2, A3 (irrelevant context)
                      But B2 can see B1 ✓
                      And B3 can see B1, B2 ✓
                      The useful signal (within-sequence) dominates
```

**Why this works in practice:**

1. **Pretraining already handles this.** During pretraining, documents are concatenated together in exactly the same way. The model has already learned to deal with boundaries between unrelated text.

2. **Loss masking helps.** If you're using completion-only loss masking, you're only computing loss on response tokens anyway. The boundary between two packed sequences typically falls between one example's `### End` and the next example's `### Instruction:` — the model isn't penalized for predicting across that boundary.

3. **The benefit outweighs the noise.** In practice, packing lets you train on **2-3x more tokens per batch** with the same GPU memory. The tiny amount of cross-sequence attention noise is negligible compared to this efficiency gain.

---

### When packing can be a problem

For **bidirectional models** (BERT, encoder-decoder) or when using **attention masking between sequences**, packing requires extra care — you'd need a **block-diagonal attention mask** to prevent sequences from attending to each other. This is more complex to implement.

---

In [26]:
def formatting_prompts_func(example):
    """Format instruction/output pairs into a single string for language modeling."""
    output_texts = []
    for i in range(len(example["instruction"])):
        text = f"### Question: {example['instruction'][i]}\n ### Answer: {example['output'][i]}"
        output_texts.append(text)
    return output_texts


def tokenize(example, dataset_text_field="text"):
    """Tokenize the text field of each example.
    # Input: {"text": str} -> Output: {"input_ids": (seq_len,), "attention_mask": (seq_len,)}
    """
    return tokenizer(example[dataset_text_field])


def pack_examples(
    examples: dict[str, list[list]], seq_length: int
) -> dict[str, list[list]]:
    """
    Data packing: concatenate all sequences then re-chunk into fixed-length blocks.
    This avoids wasting memory on padding short sequences.

    Example:
        input_ids: [[1,2,3], [4,5,6,7], [8]] with seq_length=5
        -> [[1,2,3,4,5], [6,7,8]]

    # variable-length lists -> lists of length seq_length (last may be shorter)
    """
    # Flatten all sequences into one long list
    examples = {k: sum(v, []) for k, v in examples.items()}
    # Re-chunk into blocks of seq_length
    examples = {
        k: [v[i : i + seq_length] for i in range(0, len(v), seq_length)]
        for k, v in examples.items()
    }
    return examples


def truncate(example, max_seq_length):
    """Truncate input_ids and attention_mask to max_seq_length.
    # (seq_len,) -> (min(seq_len, max_seq_length),)
    """
    return {
        key: example[key][:max_seq_length] for key in ["input_ids", "attention_mask"]
    }

In [28]:
# --- Example: tokenize, pack, and truncate ---
print(f"Is Conversational: {is_conversational(dataset['train_sft'][100])}")
dataset = dataset.map(tokenize)
print(dataset["train_sft"]["input_ids"][0][:10])

print("Sample Size Before Packing:", len(dataset["train_sft"]))
dataset = dataset.select_columns(["input_ids", "attention_mask"])
dataset = dataset.map(pack_examples, batched=True, fn_kwargs={"seq_length": 2048})
print("Sample Size After Packing:", len(dataset["train_sft"]))

dataset = dataset.map(truncate, fn_kwargs={"max_seq_length": 2048})
print("Final Sequence Length:", len(dataset["train_sft"]["input_ids"][0]))

Is Conversational: False
[151644, 8948, 198, 2610, 525, 264, 10950, 17847, 13, 151645]
Sample Size Before Packing: 115
Sample Size After Packing: 115
Final Sequence Length: 2048


## 2.3 DataCollator

Single-turn masking (mask everything up to the response key) is the collator built in [01 - Instruction Tuning](01%20-%20Instruction%20Tuning.ipynb). Below, `DataCollatorForLanguageModeling` is the no-masking baseline, and `DataCollatorForCompletionOnlyLM` adds **multi-turn** masking: when an instruction template is also given, every user turn is masked and every assistant turn is kept.
```
"### Question: What is 2+2? ### Answer: 4. ### Question: And 3+3? ### Answer: 6."
 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
 turn 1: question + template → -100        turn 2: question + template → -100
                                       ^^                                       ^^
                                       kept                                     kept
```

> **Gotcha:** template matching runs on token IDs, and BPE tokenizes the same string differently depending on what precedes it. In Qwen, `"### Question:"` at the start of a line is `['###', ' Question', ':']`, but after a space it becomes `[' ###', ...]` and silently fails to match, so that turn is never masked. Put each template after a newline, or encode the template with the same leading context it has in the data.

In [33]:
class DataCollatorForLanguageModeling:
    """
    Standard causal LM collator.
    - Pads sequences to equal length
    - Labels = input_ids (model learns to predict EVERY token)
    - Padding positions masked to -100 (ignored by loss)

    Input:  list of dicts [{"input_ids": [...], "attention_mask": [...]}, ...]
    Output: {"input_ids": (batch, seq_len),
             "attention_mask": (batch, seq_len),
             "labels": (batch, seq_len)}
    """

    def __init__(self, tokenizer, pad_to_multiple_of=8):
        self.tokenizer = tokenizer
        self.pad_to_multiple_of = pad_to_multiple_of

    def __call__(self, examples):
        input_ids = [e["input_ids"] for e in examples]
        attention_mask = [e["attention_mask"] for e in examples]

        # --- Step 1: Pad to max length in batch ---
        max_len = max(len(seq) for seq in input_ids)

        # Round up to multiple of 8 for GPU efficiency
        if self.pad_to_multiple_of:
            remainder = max_len % self.pad_to_multiple_of
            if remainder != 0:
                max_len += self.pad_to_multiple_of - remainder

        # Pad each sequence: real tokens on left, padding on right
        input_ids = [
            seq + [self.tokenizer.pad_token_id] * (max_len - len(seq))
            for seq in input_ids
        ]
        attention_mask = [seq + [0] * (max_len - len(seq)) for seq in attention_mask]

        # --- Step 2: Create labels ---
        # Labels = copy of input_ids, with padding masked to -100
        # Example:
        #   input_ids: [Hello, world, !, PAD, PAD]
        #   labels:    [Hello, world, !, -100, -100]
        labels = []
        for seq in input_ids:
            label = [tok if tok != self.tokenizer.pad_token_id else -100 for tok in seq]
            labels.append(label)

        return {
            "input_ids": torch.tensor(input_ids),
            "attention_mask": torch.tensor(attention_mask),
            "labels": torch.tensor(labels),
        }


class DataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):
    """
    Completion-only collator — inherits from DataCollatorForLanguageModeling.
    Extra step: masks instruction/prompt tokens so the model ONLY learns
    to predict the response/answer tokens.

    Two modes:
      Mode A (response_template only):  masks everything before the response
      Mode B (both templates):          handles multi-turn conversations

    Input:  same as parent
    Output: same as parent, but labels has prompt positions set to -100
    """

    def __init__(
        self,
        tokenizer,
        response_template,
        instruction_template=None,
        pad_to_multiple_of=8,
    ):
        super().__init__(tokenizer, pad_to_multiple_of)

        self.response_template = response_template
        self.instruction_template = instruction_template

        # Pre-encode templates to token IDs for matching
        self.response_token_ids = tokenizer.encode(
            response_template, add_special_tokens=False
        )
        self.instruction_token_ids = (
            tokenizer.encode(instruction_template, add_special_tokens=False)
            if instruction_template
            else None
        )

    def _find_all_occurrences(self, sequence, pattern):
        """Find all starting indices where pattern appears in sequence."""
        positions = []
        for i in range(len(sequence) - len(pattern) + 1):
            if sequence[i : i + len(pattern)] == pattern:
                positions.append(i)
        return positions

    def __call__(self, examples):
        # --- Step 1: Use parent to pad + create base labels ---
        batch = super().__call__(examples)
        labels = batch["labels"].clone()

        for i in range(len(examples)):
            seq = batch["input_ids"][i].tolist()

            if self.instruction_template is None:
                # ── Mode A: only response_template ──
                # Find LAST occurrence of response template, mask everything before it
                #
                # "### Question: What is 2+2? ### Answer: 4."
                #  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
                #  mask all of this (including "### Answer:") → -100
                #                                              ^^^^
                #                                              keep only this

                positions = self._find_all_occurrences(seq, self.response_token_ids)

                if not positions:
                    labels[i, :] = -100  # no response found, mask everything
                else:
                    last_pos = positions[-1]
                    end = last_pos + len(self.response_token_ids)
                    labels[i, :end] = -100

            else:
                # ── Mode B: both instruction + response templates ──
                # For multi-turn: mask ALL instruction segments, keep ALL response segments
                #
                # "### Q: What is 2+2? ### A: 4. ### Q: And 3+3? ### A: 6."
                #  ^^^^^^^^^^^^^^^^^^^^^^^^       ^^^^^^^^^^^^^^^^^^^^^^^
                #  Turn 1 instruction → -100      Turn 2 instruction → -100
                #                        ^^^^                            ^^^^
                #                        keep                            keep

                human_positions = self._find_all_occurrences(
                    seq, self.instruction_token_ids
                )
                response_positions = self._find_all_occurrences(
                    seq, self.response_token_ids
                )

                if not human_positions or not response_positions:
                    labels[i, :] = -100
                    continue

                # Get the end position of each response template (where actual answer starts)
                response_ends = [
                    pos + len(self.response_token_ids) for pos in response_positions
                ]

                # Mask from start to first response end
                labels[i, : response_ends[0]] = -100

                # Mask each subsequent instruction → response segment
                for h_start, r_end in zip(human_positions[1:], response_ends[1:]):
                    labels[i, h_start:r_end] = -100

                # Mask trailing instruction with no response
                if len(human_positions) > len(response_positions):
                    labels[i, human_positions[-1] :] = -100

        batch["labels"] = labels
        return batch

In [34]:
# --- Example: basic language modeling collator ---
data_collator = DataCollatorForLanguageModeling(tokenizer)
batch = data_collator([dataset["train_sft"][i] for i in range(8)])
print({k: v.shape for k, v in batch.items()})

{'input_ids': torch.Size([8, 2048]), 'attention_mask': torch.Size([8, 2048]), 'labels': torch.Size([8, 2048])}


In [ ]:
# --- Example: multi-turn completion-only collator (loss only on every answer) ---
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

response_template = "### Answer:"
instruction_template = "### Question:"
collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template,
    instruction_template=instruction_template,
    tokenizer=tokenizer,
)

# Turns are newline-separated so every "###" tokenizes identically. Mid-sentence,
# " ### Question:" becomes [' ###', ...] and no longer matches the template's ['###', ...].
text = "### Question: What is 2+2?\n### Answer: 4.\n### Question: And 3+3?\n### Answer: 6."
tokens = tokenizer(text, return_tensors="pt")
batch = collator(
    [{"input_ids": tokens["input_ids"][0].tolist(), "attention_mask": tokens["attention_mask"][0].tolist()}]
)

# Show which tokens contribute to the loss (not masked to -100)
loss_tokens = [
    tokenizer.decode([batch["input_ids"][0][i]])
    for i, label in enumerate(batch["labels"][0])
    if label != -100
]
print(f"Tokens that contribute to loss: {loss_tokens}")

## 2.4 Trainer

In [ ]:
class SimpleSFTTrainer:

    def __init__(
        self,
        model,
        tokenizer,
        train_dataset,
        eval_dataset=None,
        batch_size=4,
        learning_rate=5e-5,
        num_epochs=1,
        completion_only_loss=True,
        response_template=" ### Answer:",
        instruction_template="### Question:",
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.num_epochs = num_epochs

        # --- Optimizer ---
        self.optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

        # --- Pick the right collator ---
        # completion_only_loss=True  → only train on response tokens
        # completion_only_loss=False → train on all tokens (standard LM)
        if completion_only_loss:
            collator = DataCollatorForCompletionOnlyLM(
                tokenizer=tokenizer,
                response_template=response_template,
                instruction_template=instruction_template,
            )
        else:
            collator = DataCollatorForLanguageModeling(tokenizer=tokenizer)

        # --- DataLoaders ---
        self.train_loader = DataLoader(
            train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collator
        )
        self.eval_loader = (
            DataLoader(
                eval_dataset, batch_size=batch_size, shuffle=False, collate_fn=collator
            )
            if eval_dataset
            else None
        )

    def train(self):
        """
        The core training loop.

        Each batch goes through:
          input_ids (batch, seq_len)
            → embedding layer     → (batch, seq_len, embed_dim)
            → transformer layers  → (batch, seq_len, model_dim)
            → lm_head             → (batch, seq_len, vocab_size)
            → cross_entropy loss  → scalar
        """
        self.model.train()

        for epoch in range(self.num_epochs):
            total_loss = 0
            num_batches = 0

            for batch_idx, batch in enumerate(
                tqdm(self.train_loader, desc=f"Epoch {epoch+1}")
            ):

                # 1) Move batch to GPU
                batch = {k: v.to(self.model.device) for k, v in batch.items()}

                # 2) Forward pass — model computes logits + loss internally
                #    The model handles label shifting: logits[:-1] vs labels[1:]
                outputs = self.model(**batch)
                loss = outputs.loss

                # 3) Backward pass — compute gradients
                loss.backward()

                # 4) Update weights
                self.optimizer.step()
                self.optimizer.zero_grad()

                total_loss += loss.item()
                num_batches += 1

                if batch_idx % 10 == 0:
                    print(f"  Batch {batch_idx}, Loss: {loss.item():.4f}")

            avg_loss = total_loss / num_batches
            print(f"Epoch {epoch+1} — Avg Loss: {avg_loss:.4f}")

    def evaluate(self):
        """Run evaluation — same forward pass but no gradient computation."""
        if not self.eval_loader:
            print("No evaluation dataset provided.")
            return None

        self.model.eval()
        total_loss = 0
        num_batches = 0

        with torch.no_grad():  # disable gradients for speed + memory
            for batch in tqdm(self.eval_loader, desc="Evaluating"):
                batch = {k: v.to(self.model.device) for k, v in batch.items()}
                outputs = self.model(**batch)
                total_loss += outputs.loss.item()
                num_batches += 1

        avg_loss = total_loss / num_batches
        print(f"Eval Loss: {avg_loss:.4f}")
        return avg_loss

model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto", dtype=torch.bfloat16
)

In [40]:
test_data = [
    {"text": "### Question: What is 2+2? ### Answer: 2+2 equals 4."},
    {
        "text": "### Question: What is the capital of France? ### Answer: The capital of France is Paris."
    },
    {
        "text": "### Question: What is photosynthesis? ### Answer: Photosynthesis is the process by which plants convert sunlight into energy."
    },
]

test_dataset = Dataset.from_list(test_data)
test_dataset = test_dataset.map(tokenize, batched=True)

# Verify the completion-only collator masks instruction tokens
collator = DataCollatorForCompletionOnlyLM(
    response_template=" ### Answer:",
    instruction_template="### Question:",
    tokenizer=tokenizer,
)

batch = collator([test_dataset[0], test_dataset[1]])
print(f"Batch keys: {batch.keys()}")
# 2 variable-length sequences -> (2, max_seq_len)
print(f"Input shape: {batch['input_ids'].shape}")
print(f"Labels shape: {batch['labels'].shape}")
print(f"Masked tokens (-100): {(batch['labels'] == -100).sum().item()}")
print(f"Loss tokens: {(batch['labels'] != -100).sum().item()}")

Batch keys: dict_keys(['input_ids', 'attention_mask', 'labels'])
Input shape: torch.Size([2, 24])
Labels shape: torch.Size([2, 24])
Masked tokens (-100): 33
Loss tokens: 15


In [22]:
trainer = SimpleSFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=test_dataset,
    batch_size=1,
    learning_rate=1e-4,
    num_epochs=1,
    completion_only_loss=True,
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
trainer.train()

Model parameters: 494,032,768



Epoch 1:   0%|          | 0/3 [00:00<?, ?it/s]


Epoch 1:  33%|███▎      | 1/3 [00:04<00:08,  4.25s/it]

  Batch 0, Loss: 1.0379



Epoch 1:  67%|██████▋   | 2/3 [00:05<00:02,  2.34s/it]


Epoch 1: 100%|██████████| 3/3 [00:05<00:00,  1.38s/it]


Epoch 1: 100%|██████████| 3/3 [00:05<00:00,  1.83s/it]

Epoch 1 — Avg Loss: 1.3757


# 3) Label Right-Shifting

A common question: **"Do I need to shift my labels so that position `i` predicts position `i+1`?"**

**No.** The data collator outputs `labels = input_ids` (same length, same positions). The **model itself** handles the right-shift internally during loss computation.

### Where it happens

This code lives inside the model's `forward()` method (e.g., `transformers/models/qwen2/modeling_qwen2.py`). Every HuggingFace causal LM model has this same pattern:
```python
if labels is not None:
    # Shift so that token at position i predicts token at position i+1
    shift_logits = logits[..., :-1, :].contiguous()   # drop last
    shift_labels = labels[..., 1:].contiguous()         # drop first

    # Flatten for cross-entropy: (batch × seq_len) predictions vs targets
    loss_fct = CrossEntropyLoss()
    shift_logits = shift_logits.view(-1, self.config.vocab_size)
    shift_labels = shift_labels.view(-1)

    loss = loss_fct(shift_logits, shift_labels)
```